# 03 — Autoencoder

Autoencoder training and reduced features. Uses `src.models.autoencoder` and `src.features`.

In [1]:
# Path fix: use this repo's src
from pathlib import Path
import sys
import logging
import numpy as np
import torch
PROJECT_ROOT = Path.cwd().resolve().parents[0]
project_root_str = str(PROJECT_ROOT)
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)
for name in list(sys.modules.keys()):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

from src.utils import SEED, set_global_seed
from src.utils.paths import get_multirocket_features_dir, get_reduced_dir, get_models_dir, ensure_dir
from src.features.memmap import open_memmap_read
from src.features.scaling import fit_scaler, save_scaler, load_scaler, transform_with_scaler
from src.models.autoencoder import train_autoencoder, save_autoencoder

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

# Autoencoder params (match experiments/A1_autoencoder_mlp and A2_autoencoder_ft)
LATENT_DIM = 256
HIDDEN_DIMS = [512, 256]
AE_EPOCHS = 50
AE_BATCH_SIZE = 256
AE_LR = 1e-3
print("AE: latent_dim=%s hidden_dims=%s epochs=%s" % (LATENT_DIM, HIDDEN_DIMS, AE_EPOCHS))

AE: latent_dim=256 hidden_dims=[512, 256] epochs=50


In [2]:
# Load MultiROCKET feature shapes and data
mr_dir = get_multirocket_features_dir()
shapes_path = mr_dir / "shapes.npz"
if not shapes_path.exists():
    raise FileNotFoundError("Run 02_multirocket.ipynb first to create %s" % shapes_path)
shapes = np.load(shapes_path)
train_shape = tuple(shapes["train"])
test_shape = tuple(shapes["test"])
val_shape = tuple(shapes["val"]) if "val" in shapes else None
print("Loading MultiROCKET features...")
F_train = np.array(open_memmap_read(mr_dir / "train.dat", train_shape), dtype=np.float32)
F_test = np.array(open_memmap_read(mr_dir / "test.dat", test_shape), dtype=np.float32)
if val_shape:
    F_val = np.array(open_memmap_read(mr_dir / "val.dat", val_shape), dtype=np.float32)
else:
    F_val = None
n_features = F_train.shape[1]
print("F_train %s F_test %s n_features=%d" % (F_train.shape, F_test.shape, n_features))

Loading MultiROCKET features...
F_train (36120, 16128) F_test (6773, 16128) n_features=16128


In [3]:
# Scale features: fit on train, transform train/val/test
scaler_path = get_models_dir() / "scalers" / "scaler.joblib"
if scaler_path.exists():
    print("Loading scaler from %s..." % scaler_path)
    scaler = load_scaler(scaler_path)
else:
    print("Fitting scaler on train...")
    scaler = fit_scaler(F_train)
    ensure_dir(scaler_path.parent)
    save_scaler(scaler, scaler_path)
print("Transforming train/val/test...")
F_train = np.asarray(transform_with_scaler(scaler, F_train), dtype=np.float32)
F_test = np.asarray(transform_with_scaler(scaler, F_test), dtype=np.float32)
if F_val is not None:
    F_val = np.asarray(transform_with_scaler(scaler, F_val), dtype=np.float32)
print("Scaled features ready.")

Fitting scaler on train...


2026-02-28 13:21:19,340 | INFO | Fitted StandardScaler on shape (36120, 16128)
2026-02-28 13:21:19,345 | INFO | Saved scaler to C:\Projects\DeepLearningProject\models\scalers\scaler.joblib


Transforming train/val/test...
Scaled features ready.


In [4]:
# Train autoencoder on scaled train features
set_global_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: %s" % device)
t_train = torch.from_numpy(F_train).float().to(device)
print("Training autoencoder (%d epochs)..." % AE_EPOCHS)
encoder, _ = train_autoencoder(
    t_train,
    input_dim=n_features,
    hidden_dims=HIDDEN_DIMS,
    latent_dim=LATENT_DIM,
    epochs=AE_EPOCHS,
    batch_size=AE_BATCH_SIZE,
    lr=AE_LR,
    device=device,
)
print("Training complete.")

2026-02-28 13:21:23,084 | INFO | Global seed set to 0


Device: cpu
Training autoencoder (50 epochs)...


2026-02-28 13:21:37,622 | INFO | AE epoch 1 loss 0.529695
2026-02-28 13:23:48,688 | INFO | AE epoch 10 loss 0.326922
2026-02-28 13:26:22,195 | INFO | AE epoch 20 loss 0.297124
2026-02-28 13:28:58,318 | INFO | AE epoch 30 loss 0.276544
2026-02-28 13:31:39,868 | INFO | AE epoch 40 loss 0.294730
2026-02-28 13:34:22,834 | INFO | AE epoch 50 loss 0.269604


Training complete.


In [5]:
# Save encoder and compute reduced (latent) features
ae_path = get_models_dir() / "autoencoders" / "encoder.pt"
ensure_dir(ae_path.parent)
print("Saving encoder to %s..." % ae_path)
save_autoencoder(encoder, ae_path)

encoder.eval()
with torch.no_grad():
    Z_train = encoder(t_train).cpu().numpy()
    Z_test = encoder(torch.from_numpy(F_test).float().to(device)).cpu().numpy()
    if F_val is not None:
        Z_val = encoder(torch.from_numpy(F_val).float().to(device)).cpu().numpy()
    else:
        Z_val = None
print("Reduced: Z_train %s Z_test %s" % (Z_train.shape, Z_test.shape))

2026-02-28 13:34:22,905 | INFO | Saved autoencoder to C:\Projects\DeepLearningProject\models\autoencoders\encoder.pt


Saving encoder to C:\Projects\DeepLearningProject\models\autoencoders\encoder.pt...
Reduced: Z_train (36120, 256) Z_test (6773, 256)


In [6]:
# Save reduced features to data/processed/reduced/autoencoder/ (used by train.py for A1/A2)
out_dir = ensure_dir(get_reduced_dir() / "autoencoder")
print("Saving to %s..." % out_dir)
np.save(out_dir / "train.npy", Z_train.astype(np.float32))
np.save(out_dir / "test.npy", Z_test.astype(np.float32))
if Z_val is not None:
    np.save(out_dir / "val.npy", Z_val.astype(np.float32))
print("Done. Autoencoder reduced features saved (%d -> %d dims)." % (n_features, LATENT_DIM))

Saving to C:\Projects\DeepLearningProject\data\processed\reduced\autoencoder...
Done. Autoencoder reduced features saved (16128 -> 256 dims).
